# End of week 1 exercise

To demonstrate familiarity with OpenAI API, Ollama, and also Claude build a tool that takes a technical question,  
and responds with an explanation. 

In [ ]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
from anthropic import Anthropic

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
MODEL_CLAUDE = 'claude-sonnet-5'

In [ ]:
# set up environment

load_dotenv(override=True)
api_key_openai = os.getenv('OPENAI_API_KEY')

if api_key_openai and api_key_openai.startswith('sk-proj-') and len(api_key_openai)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")    

openai_client = OpenAI(api_key=api_key_openai)

# set up ollama client
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# set up anthropic client
api_key_claude = os.getenv('ANTHROPIC_API_KEY')
claude_client = Anthropic()

In [ ]:
system_prompt = """ 
You are a mentor and specialized in programming. Your responsibility is to help the junior programmer to learn the code provided in a very easy details. Assume that they have no prior knowledge about coding."""

In [ ]:
def get_user_prompt(code):
    user_prompt = f""" Please explain what this code does and why: {code}"""
    return user_prompt

In [ ]:
# Get gpt-4o-mini to answer, with streaming

# def display_code_explaination(code):    
#     response = openai.chat.completions.create(
#         model=MODEL_GPT,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user", "content": get_user_prompt(code)}
#         ]
#     )
#     result = response.choices[0].message.content
#     display(Markdown(result))

def stream_code_explaination_with_openai(code):
    stream = openai_client.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(code)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_code_explaination_with_openai('yield from {book.get("author") for book in books if book.get("author")}')

In [ ]:
# Get Llama 3.2 to answer

def stream_code_explaination_with_llama(code):
    stream = ollama_client.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_user_prompt(code)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
stream_code_explaination_with_llama('yield from {book.get("author") for book in books if book.get("author")}')

In [ ]:
# Get claude-sonnet-5 to answer

def stream_code_explaination_with_claude(code):
    stream = claude_client.messages.create(
      model=MODEL_CLAUDE,
      max_tokens=1024,
      system=system_prompt,
      messages=[
          {"role": "user", "content": get_user_prompt(code)}
        ],
      stream=True
    ) 
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
      if chunk.type == "content_block_delta" and chunk.delta.type == "text_delta":
        response += chunk.delta.text
        update_display(Markdown(response), display_id=display_handle.display_id)   

In [ ]:
stream_code_explaination_with_claude('yield from {book.get("author") for book in books if book.get("author")}')